In [1]:
import sys
from pathlib import Path

import pandas as pd
import surprise
from surprise import accuracy

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parents[1]
elif project_root.name == "tarea_01":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tarea_01.experiments_item_funksvd import build_ranking_testset  # noqa: E402
from tarea_01.utils import calculate_metrics  # noqa: E402

In [2]:
dataset_path = Path('../dataset')
train_df = pd.read_csv(dataset_path / 'train.csv')
train_df.date = pd.to_datetime(train_df.date)

test_df = pd.read_csv(dataset_path / 'validation.csv')
test_df.date = pd.to_datetime(test_df.date)

reader = surprise.reader.Reader(rating_scale=(0, 5))

train_data = surprise.Dataset.load_from_df(
    train_df[["user_id", "recipe_id", "rating"]], reader=reader
)
trainset = train_data.build_full_trainset()

test_data = surprise.Dataset.load_from_df(
    test_df[["user_id", "recipe_id", "rating"]], reader=reader
)
testset = test_data.construct_testset(test_data.raw_ratings)

In [3]:
factor_values = [20, 50, 100]
lr_values = [0.003, 0.005]
reg_values = [0.02, 0.05]

rows = []
models = {}

for factors in factor_values:
    for lr_all in lr_values:
        for reg_all in reg_values:
            model = surprise.SVD(
                n_factors=factors,
                n_epochs=30,
                lr_all=lr_all,
                reg_all=reg_all,
                biased=True,
                random_state=42,
            )
            model.fit(trainset)
            predictions = model.test(testset)
            rmse = accuracy.rmse(predictions, verbose=False)
            mae = accuracy.mae(predictions, verbose=False)
            rows.append(
                {
                    "method": "FunkSVD",
                    "n_factors": factors,
                    "lr_all": lr_all,
                    "reg_all": reg_all,
                    "biased": True,
                    "RMSE": round(float(rmse), 4),
                    "MAE": round(float(mae), 4),
                }
            )
            models[(factors, lr_all, reg_all)] = model

funksvd_results = pd.DataFrame(rows)
funksvd_results.sort_values("RMSE")

,method,n_factors,lr_all,reg_all,biased,RMSE,MAE
1,FunkSVD,20,0.003,0.05,True,0.8841,0.4813
5,FunkSVD,50,0.003,0.05,True,0.8849,0.4825
0,FunkSVD,20,0.003,0.02,True,0.8851,0.4812
9,FunkSVD,100,0.003,0.05,True,0.8863,0.4840
4,FunkSVD,50,0.003,0.02,True,0.8868,0.4833
3,FunkSVD,20,0.005,0.05,True,0.8873,0.4808
8,FunkSVD,100,0.003,0.02,True,0.8887,0.4854
7,FunkSVD,50,0.005,0.05,True,0.8887,0.4827
11,FunkSVD,100,0.005,0.05,True,0.8900,0.4839
2,FunkSVD,20,0.005,0.02,True,0.8905,0.4820


## Mejor configuracion

Se selecciona por menor RMSE en validacion para mantener el mismo criterio usado en los notebooks de vecinos.

In [4]:
best_row = funksvd_results.sort_values("RMSE").iloc[0]
best_key = (
    int(best_row["n_factors"]),
    float(best_row["lr_all"]),
    float(best_row["reg_all"]),
)
best_model = models[best_key]
best_row

method       FunkSVD
n_factors         20
lr_all         0.003
reg_all         0.05
biased          True
RMSE          0.8841
MAE           0.4813
Name: 1, dtype: object

## Ranking Top-10

Para ranking se agregan negativos muestreados al set de validacion. El RMSE puro se toma desde la tabla anterior; `RankingSetRMSE` solo describe el set artificial usado para rankear.

In [5]:
ranking_negatives = 100
# La seed 42 no viene impuesta por el enunciado; se fija para reproducibilidad.
ranking_testset = build_ranking_testset(
    train_df=train_df,
    validation_df=test_df,
    n_negatives=ranking_negatives,
    seed=42,
)

metrics = calculate_metrics(
    model=best_model,
    testset=ranking_testset,
    k=10,
    threshold=4.0,
)
ranking_set_rmse = metrics.pop("RMSE")

pd.DataFrame(
    {
        "FunkSVD": {
            "RMSE": best_row["RMSE"],
            "MAE": best_row["MAE"],
            "RankingSetRMSE": ranking_set_rmse,
            **metrics,
        }
    }
).T

,RMSE,MAE,RankingSetRMSE,Recall@10,Precision@10,NDCG@10,MAP@10,Diversity,Novelty
FunkSVD,0.8841,0.4813,4.5055,0.1634,0.0969,0.1873,0.1295,0.9842,12.9197
